In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer,StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score,GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error,root_mean_squared_error,mean_squared_error

In [2]:
df = pd.read_csv('cleaned_engineered.csv')

In [3]:
skew_num = ['Study_Hours']
other_num = ['Age','Avg_Daily_Usage_Hours','Daily_Unlocks','Physical_Activity_Hours','Sleep_Hours_Per_Night']
ord_cat = ['Stress_Level','Academic_Level']
ohe_cat = ['Gender','Country','Most_Used_Platform','Purpose_Of_Use']
cols = skew_num+other_num+ord_cat+ohe_cat
X = df[cols]
y = df['Mental_Health_Score']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [4]:
#1. Skewed features
skew_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())

])

#2. Numeric Features
plain_numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])

#3. Ordinal
ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High'],['High School','Undergraduate','Graduate',]]))
])

#4. Nominal Features
nominal_pipeline = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown="ignore",drop='first'))
])


preprocessor = ColumnTransformer(transformers=[
    ("Skewed_Pipeline", skew_pipeline, skew_num),
    ("Plain_Numeric",plain_numeric_pipeline, other_num ),
    ('Ordinal', ordinal_pipeline, ord_cat),
    ('Normal', nominal_pipeline, ohe_cat)
])


In [5]:
tree_pipe = Pipeline([['preprocessor',preprocessor],['tree',RandomForestRegressor()]])

In [6]:
tree_pipe.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ['tree', RandomForestRegressor()]]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Skewed_Pipeline', ...), ('Plain_Numeric', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [8]:
y_pred_test = tree_pipe.predict(X_test)
y_pred_train = tree_pipe.predict(X_train)

In [9]:
print("Test r2",r2_score(y_test,y_pred_test))
print("Train r2",r2_score(y_train,y_pred_train))
print ("Test MAE", mean_absolute_error(y_test,y_pred_test))
print("Test RMSE",root_mean_squared_error(y_test,y_pred_test))

Test r2 0.9178120625141302
Train r2 0.9870646672463967
Test MAE 0.2768626428571429
Test RMSE 0.3830438312430376


In [11]:
scores = cross_val_score(tree_pipe,X,y,cv=5,scoring='r2')

In [12]:
scores.mean()

np.float64(0.9162476931569685)

In [30]:
params = {'tree__n_estimators' : [500,700,900] , 'tree__max_depth' : [20,25],'tree__min_samples_split' : [2,3]
          ,'tree__min_samples_leaf':[1,3],'tree__max_features':[0.5,0.75,1]}

In [31]:
grid = GridSearchCV(tree_pipe,param_grid=params,cv=3,n_jobs=-1,scoring='r2',verbose=3)

In [32]:
grid.fit(X_train,y_train)

Fitting 3 folds for each of 72 candidates, totalling 216 fits

[CV 2/3] END tree__max_depth=7, tree__max_features=1, tree__min_samples_leaf=7, tree__min_samples_split=5, tree__n_estimators=300;, score=0.483 total time=   0.3s
[CV 1/3] END tree__max_depth=7, tree__max_features=1, tree__min_samples_leaf=7, tree__min_samples_split=10, tree__n_estimators=100;, score=0.530 total time=   0.1s
[CV 3/3] END tree__max_depth=7, tree__max_features=1, tree__min_samples_leaf=7, tree__min_samples_split=10, tree__n_estimators=200;, score=0.470 total time=   0.2s
[CV 1/3] END tree__max_depth=10, tree__max_features=0.5, tree__min_samples_leaf=1, tree__min_samples_split=2, tree__n_estimators=200;, score=0.861 total time=   1.6s
[CV 1/3] END tree__max_depth=10, tree__max_features=0.5, tree__min_samples_leaf=1, tree__min_samples_split=3, tree__n_estimators=200;, score=0.860 total time=   1.6s
[CV 2/3] END tree__max_depth=10, tree__max_features=0.5, tree__min_samples_leaf=1, tree__min_samples_split=5, tree

,estimator,Pipeline(step...Regressor()]])
,param_grid,"{'tree__max_depth': [20, 25], 'tree__max_features': [0.5, 0.75, ...], 'tree__min_samples_leaf': [1, 3], 'tree__min_samples_split': [2, 3], ...}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,3
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('Skewed_Pipeline', ...), ('Plain_Numeric', ...), ...]"


In [33]:
grid.best_score_

np.float64(0.8872631813879579)

In [34]:
grid.best_params_

{'tree__max_depth': 25,
 'tree__max_features': 0.5,
 'tree__min_samples_leaf': 1,
 'tree__min_samples_split': 2,
 'tree__n_estimators': 700}

## Random Forest Regression – Model Evaluation

Random Forest Regression was implemented to capture possible non-linear relationships between the input features and the Mental Health Score. Unlike Linear Regression, Random Forest does not assume a linear relationship between the predictors and the target variable. It combines multiple decision trees and averages their predictions to obtain the final regression output.

A baseline Random Forest model was first trained before performing hyperparameter tuning. The baseline model achieved approximately **0.91 R² on the test set** and approximately **0.91 mean 5-fold cross-validation R²**. The training R² was approximately **0.987**, while the test MAE was approximately **0.27**.

The high training R² compared with the test R² indicates that the Random Forest model fits the training data very strongly and exhibits some degree of overfitting. However, the test and cross-validation scores remained high, indicating that the model still generalizes well to unseen data.

### Hyperparameter Tuning

GridSearchCV with 5-fold cross-validation was then used to search for a better combination of Random Forest hyperparameters. The best combination found was:

- `max_depth = 25`
- `max_features = 0.5`
- `min_samples_leaf = 1`
- `min_samples_split = 2`
- `n_estimators = 700`

The best cross-validation R² obtained during hyperparameter tuning was approximately **0.8873**.

Interestingly, this score was lower than the approximately **0.91 CV R² obtained by the baseline Random Forest model**. This demonstrates that hyperparameter tuning does not necessarily guarantee an improvement in model performance. GridSearchCV only identifies the best-performing combination among the hyperparameter values provided in the search space.

Since the baseline Random Forest achieved better cross-validation performance than the tuned configurations explored, the baseline model is preferred for this dataset.

### Results

| Model | CV R² | Test R² | Train R² | Test MAE |
|---|---:|---:|---:|---:|
| Baseline Random Forest | ≈ 0.91 | ≈ 0.91 | ≈ 0.987 | ≈ 0.27 |
| Tuned Random Forest | 0.8873 | — | — | — |

### Conclusion

The baseline Random Forest Regression model performed strongly and substantially outperformed the linear models evaluated previously. Although hyperparameter tuning was performed, none of the configurations explored improved upon the baseline cross-validation performance.

Therefore, the **baseline Random Forest model is retained as the preferred Random Forest configuration** for this dataset.

The model will be compared with **Support Vector Regression and XGBoost Regression** using consistent evaluation metrics to determine the overall best-performing model.